# Rekurentné neurónové siete

V predchádzajúcom module sme prebrali bohaté sémantické reprezentácie textu. Architektúra, ktorú používame, zachytáva zoskupený význam slov vo vete, no nezohľadňuje **poradie** slov, pretože agregácia, ktorá nasleduje po embeddingoch, odstráni tieto informácie z pôvodného textu. Pretože tieto modely nedokážu reprezentovať poradie slov, nemôžu riešiť zložitejšie alebo nejednoznačné úlohy ako generovanie textu alebo odpovedanie na otázky.

Na zachytenie významu textovej sekvencie použijeme neurónovú sieť nazývanú **rekurentná neurónová sieť**, alebo RNN. Pri použití RNN posielame našu vetu do siete po jednom tokene a sieť vytvára určitý **stav**, ktorý následne posielame späť do siete spolu s ďalším tokenom.

![Obrázok ukazujúci príklad generovania v rekuretnej neurónovej sieti.](../../../../../translated_images/sk/rnn.27f5c29c53d727b5.webp)

Vzhľadom na vstupnú sekvenciu tokenov $X_0,\dots,X_n$ vytvára RNN sekvenciu neurónových blokov a túto sekvenciu trénuje end-to-end použitím spätného šírenia chyby. Každý neurónový blok ako vstup prijíma dvojicu $(X_i,S_i)$ a ako výsledok produkuje $S_{i+1}$. Konečný stav $S_n$ alebo výstup $Y_n$ vstupuje do lineárneho klasifikátora, ktorý vytvára výsledok. Všetky bloky majú zdieľané rovnaké váhy a trénujú sa end-to-end jedným priechodom spätného šírenia.

> Obrázok vyššie ukazuje rekurentnú neurónovú sieť v rozvinutom tvare (vľavo) a v kompaktnejšom tvare rekuretnej reprezentácie (vpravo). Je dôležité uvedomiť si, že všetky RNN bunky majú rovnaké **zdieľateľné váhy**.

Pretože stavové vektory $S_0,\dots,S_n$ sú prenášané cez sieť, RNN dokáže učiť postupné závislosti medzi slovami. Napríklad, keď sa v sekvencii objaví slovo *not*, dokáže sa naučiť negatívne ovplyvniť určité prvky vo vektorovom stave.

Vnútri každá RNN bunka obsahuje dve váhové matice: $W_H$ a $W_I$, a prah $b$. V každom RNN kroku, vzhľadom na vstup $X_i$ a vstupný stav $S_i$, sa výstupný stav vypočíta ako $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$, kde $f$ je aktivačná funkcia (často $\tanh$).

> Pre úlohy ako generovanie textu (ktoré preberieme v ďalšej jednotke) alebo strojový preklad chceme tiež získať nejakú výstupnú hodnotu v každom kroku RNN. V tomto prípade existuje ďalšia matica $W_O$ a výstup sa vypočíta ako $Y_i=f(W_O\times S_i+b_O)$.

Pozrime sa, ako nám rekuretne neurónové siete môžu pomôcť klasifikovať náš dataset správ.

> Pre sandboxové prostredie je potrebné spustiť nasledujúcu bunku, aby sa overilo, že potrebná knižnica je nainštalovaná a dáta sú prednačítané. Ak bežíte lokálne, túto bunku môžete preskočiť.


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

Pri trénovaní veľkých modelov môže byť problémom alokácia pamäte GPU. Tiež môžeme potrebovať experimentovať s rôznymi veľkosťami minibatchov, aby sa dáta zmestili do našej pamäte GPU a zároveň bolo trénovanie dostatočne rýchle. Ak spúšťate tento kód na vlastnom GPU zariadení, môžete experimentovať s úpravou veľkosti minibatchu na zrýchlenie trénovania.

> **Poznámka**: Niektoré verzie ovládačov NVidia sú známe tým, že neuvoľňujú pamäť po trénovaní modelu. V tomto zápisníku spúšťame niekoľko príkladov, čo môže v určitých nastaveniach spôsobiť vyčerpanie pamäte, najmä ak robíte vlastné experimenty v rámci toho istého zápisníka. Ak sa vám pri začiatku trénovania modelu vyskytnú nejaké zvláštne chyby, môže pomôcť reštartovanie kernelu zápisníka.


In [3]:
batch_size = 16
embed_size = 64

## Jednoduchý RNN klasifikátor

V prípade jednoduchého RNN je každá rekurentná jednotka jednoduchá lineárna sieť, ktorá prijíma vstupný vektor a stavový vektor a produkuje nový stavový vektor. V Keras to môže byť reprezentované vrstvou `SimpleRNN`.

Hoci môžeme priamo odovzdať do vrstvy RNN tokeny zakódované one-hot, nie je to dobrý nápad kvôli ich vysokej dimenzionalite. Preto použijeme embedding vrstvu na zníženie dimenzie vektorov slov, následne vrstvu RNN a nakoniec klasifikátor `Dense`.

> **Poznámka**: V prípadoch, keď dimenzionalita nie je tak vysoká, napríklad pri tokenizácii na úrovni znakov, môže dávať zmysel priamo prenášať one-hot kódované tokeny do RNN bunky.


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **Poznámka:** Pre jednoduchosť používame neučenú embedding vrstvu, ale pre lepšie výsledky môžeme použiť predtrénovanú embedding vrstvu pomocou Word2Vec, ako je popísané v predchádzajúcej jednotke. Bolo by to dobré cvičenie pre vás, upraviť tento kód tak, aby pracoval s predtrénovanými embeddingmi.

Teraz si natrénujeme našu RNN. RNN sú všeobecne dosť náročné na trénovanie, pretože keď sú RNN bunky odvinuté pozdĺž dĺžky sekvencie, výsledný počet vrstiev zapojených do spätného šírenia je dosť veľký. Preto potrebujeme zvoliť nižšiu mieru učenia a trénovať sieť na väčšom dátovom súbore, aby sme dosiahli dobré výsledky. To môže trvať pomerne dlho, preto je preferované používať GPU.

Na zrýchlenie budeme trénovať model RNN iba na tituloch správ, vynecháme popisy. Môžete skúsiť trénovať aj s popisom a uvidíte, či sa vám podarí model natrénovať.


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **Poznámka**: Presnosť bude pravdepodobne nižšia, pretože trénujeme iba na titulkoch správ.


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


Keďže teraz používame maskovanie, môžeme trénovať model na celej dátovej sade titulov a popisov.

> **Poznámka**: Všimli ste si, že sme používali vektorizér trénovaný na titulkoch správ, a nie na celom texte článku? Potenciálne to môže spôsobiť, že niektoré tokeny budú ignorované, preto je lepšie vektorizér pretrénovať. Môže však mať len veľmi malý vplyv, takže pre jednoduchosť zostaneme pri predchádzajúcom predtrénovanom vektorizéri.


## LSTM: Dlhodobá krátkodobá pamäť

Jedným z hlavných problémov RNN je **miznúci gradient**. RNN môžu byť pomerne dlhé a môže byť pre ne ťažké šíriť gradienty späť až k prvej vrstve siete počas spätného šírenia. Keď sa to stane, sieť sa nemôže naučiť vzťahy medzi vzdialenými tokenmi. Jedným zo spôsobov, ako sa tomuto problému vyhnúť, je zavedenie **explicitného riadenia stavu** pomocou **brán**. Dve najbežnejšie architektúry, ktoré zavádzajú brány, sú **dlhodobá krátkodobá pamäť** (LSTM) a **gated relay unit** (GRU). Tu sa budeme venovať LSTM.

![Obrázok ukazujúci príklad bunky dlhotrvajúcej krátkodobej pamäti](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

Sieť LSTM je usporiadaná podobne ako RNN, ale existujú dva stavy, ktoré sa odovzdávajú z vrstvy do vrstvy: aktuálny stav $c$ a skrytý vektor $h$. V každej jednotke sa skrytý vektor $h_{t-1}$ kombinuje so vstupom $x_t$ a spoločne riadia, čo sa stane so stavom $c_t$ a výstupom $h_{t}$ cez **brány**. Každá brána má sigmoidnú aktiváciu (výstup v rozsahu $[0,1]$), ktorú je možné považovať za bitovú masku, keď sa násobí so stavovým vektorom. LSTM majú nasledujúce brány (zľava doprava na obrázku vyššie):
* **brána zabudnutia**, ktorá určuje, ktoré komponenty vektora $c_{t-1}$ potrebujeme zabudnúť a ktoré prejsť ďalej. 
* **vstupná brána**, ktorá určuje, koľko informácií zo vstupného vektora a predchádzajúceho skrytého vektora by sa malo začleniť do stavového vektora.
* **výstupná brána**, ktorá berie nový stavový vektor a rozhoduje, ktoré jeho komponenty sa použijú na vytvorenie nového skrytého vektora $h_t$.

Komponenty stavu $c$ možno považovať za príznaky, ktoré je možné zapínať a vypínať. Napríklad, keď v sekvencii narazíme na meno *Alice*, predpokladáme, že ide o ženu a zdvihneme príznak v stave, ktorý hovorí, že v vete máme ženské podstatné meno. Keď sa ďalej stretávame so slovami *and Tom*, zdvihneme príznak, ktorý hovorí, že máme množné číslo podstatného mena. Manipuláciou so stavom teda môžeme sledovať gramatické vlastnosti vety.

> **Poznámka**: Tu je skvelý zdroj na pochopenie vnútorného fungovania LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) od Christophera Olaha.

Hoci vnútorná štruktúra bunky LSTM môže vyzerať zložito, Keras skrýva túto implementáciu vo vrstve `LSTM`, takže v príklade vyššie stačí len nahradiť rekurentnú vrstvu:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **Poznámka**: Tréning LSTM je tiež pomalý a na začiatku tréningu nemusí byť viditeľné veľké zlepšenie presnosti. Môže byť potrebné pokračovať v tréningu určitý čas, aby sa dosiahla dobrá presnosť.


## Obousmerné a viacvrstvové RNN

V našich príkladoch doteraz rekurentné siete pracovali od začiatku sekvencie až do jej konca. Zdá sa nám to prirodzené, pretože to zodpovedá smeru, ktorým čítame alebo počúvame reč. Avšak v scénaroch, ktoré vyžadujú náhodný prístup k vstupnej sekvencii, má zmysel spustiť rekurentnú výpočtovú operáciu v oboch smeroch. RNN, ktoré umožňujú výpočty v oboch smeroch, sa nazývajú **obousmerné** RNN a môžu byť vytvorené zabalením rekurentnej vrstvy do špeciálnej vrstvy `Bidirectonal`.

> **Poznámka**: Vrstva `Bidirectional` vytvorí dve kópie vrstvy v sebe a nastaví vlastnosť `go_backwards` jednej z týchto kópií na hodnotu `True`, čím spôsobí, že pôjde opačným smerom pozdĺž sekvencie.

Rekurentné siete, jednoperspektívne alebo obousmerné, zachytávajú vzory v rámci sekvencie a ukladajú ich do stavových vektorov alebo ich vracajú ako výstup. Rovnako ako pri konvolučných sieťach, môžeme zostaviť ďalšiu rekurentnú vrstvu za prvou, aby sme zachytili vzory vyššej úrovne, vychádzajúce z nižších úrovní vzorov získaných prvou vrstvou. To nás vedie k pojmu **viacvrstvová RNN**, ktorá pozostáva z dvoch alebo viacerých rekurentných sietí, kde výstup z predchádzajúcej vrstvy je odovzdaný ako vstup do nasledujúcej vrstvy.

![Obrázok ukazujúci viacvrstvovú dlhodobú krátkodobú pamäť - RNN](../../../../../translated_images/sk/multi-layer-lstm.dd975e29bb2a59fe.webp)

*Obrázok z [tohto úžasného článku](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) od Fernanda Lópeza.*

Keras umožňuje jednoducho stavať tieto siete, pretože stačí pridať viac rekurentných vrstiev do modelu. Pre všetky vrstvy okrem poslednej je potrebné špecifikovať parameter `return_sequences=True`, pretože potrebujeme, aby vrstva vracala všetky medzistavy a nie iba konečný stav rekurentného výpočtu.

Postavme si dvojvrstvové obousmerné LSTM pre náš klasifikačný problém.

> **Poznámka** tento kód opäť trvá pomerne dlho, kým sa dokončí, ale poskytuje najvyššiu presnosť, akú sme zatiaľ videli. Možno teda stojí za to počkať a vidieť výsledok.


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNN pre iné úlohy

Doteraz sme sa zameriavali na použitie RNN na klasifikáciu sekvencií textu. Ale môžu zvládnuť aj mnoho ďalších úloh, ako je generovanie textu a strojový preklad &mdash; tieto úlohy zvážime v nasledujúcej jednotke.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
